# 占術ノートブック（Gemini Flash × Kaggle）

**実行手順**：① Cell 0 → ② Cell 0.1（セルフテスト）→ ③ Cell 1（CSV入力＋MTS Stage A）→ ④ Cell N2（MTS Stage B/C＋HTML）

**注意**：
- 入力は **people_sample.csv** だけです（複数人可・このセルで全員分を処理）。
- アクセラレータは **None（CPU）** でOK（GPU不要・T4枠を消費しません）
- 右パネル → Add-ons → Secrets に **GOOGLE_API_KEY** を登録してください（https://aistudio.google.com で無料発行）
- fortune_dataset を Add Input で追加、または /kaggle/working に配置してください。

In [ ]:
# ============================================================
# Cell 0: ブートストラップ（環境設定・Gemini API バックエンド）
# ※ 初回実行は1〜3分です（計算ライブラリ導入のみ。モデルのダウンロードは無し）
# ============================================================
# ★ 2026-08-26 Gemini API（Flash系・無料枠）へ完全移行した。
#   - GPU アクセラレータは「None（CPU）」でOK（T4枠を消費しない）
#   - HF_TOKEN は不要 → GOOGLE_API_KEY（Kaggle Secrets）が必要
#   - ローカルモデルのロード・ダウンロードがなく、全体が軽量
import importlib.util
import json
import os
import sys
from pathlib import Path

# ============================================================
# STEP 1: 依存ライブラリの導入（すべて import より前に完了させる）
# ============================================================
# 存在確認は find_spec（モジュールをロードしない・安全）
_pip_targets = []  # 後で一括インストールする

# --- 占術計算ライブラリ ---
_calc_libs = {
    "sxtwl": "sxtwl",               # 暦（四柱・節気）
    "swisseph": "pyswisseph",       # 占星術・HD・真太陽時（import名≠pip名）
    "iztro_py": "iztro-py",         # 紫微斗数
    "divicast": "divicast",         # 六爻
    "ephem": "ephem",               # 奇門（kinqimen依存）
    "kinqimen": None,               # 奇門（--no-deps で別途）
    "lunar_python": "lunar_python", # 算命学
    "geopy": "geopy",               # HD（都市→座標）
    "timezonefinder": "timezonefinder",
    "pytz": "pytz",
}
for _mod, _pip in _calc_libs.items():
    if importlib.util.find_spec(_mod) is None:
        if _pip:
            _pip_targets.append(_pip)
        print("  不足:", _mod, "→ pip:", _pip)
if _pip_targets:
    print("ライブラリ導入:", _pip_targets)
    os.system("pip install -q " + " ".join(_pip_targets))
if importlib.util.find_spec("kinqimen") is None:
    print("  不足: kinqimen → pip (--no-deps)")
    os.system("pip install -q --no-deps kinqimen")

# --- Gemini API SDK（google-genai）---
# ★ pip名は google-genai・import名は google.genai（find_spec で判定）
if importlib.util.find_spec("google.genai") is None:
    print("  不足: google.genai → pip: google-genai")
    os.system("pip install -q google-genai")

# ============================================================
# STEP 2: fortunelib の読込（自動検出版・再帰）
# ============================================================
def _find_fortunelib(input_root="/kaggle/input", working_root="/kaggle/working",
                     max_depth=6):
    """fortunelib を含むベースディレクトリを探す（再帰・深さ制限付き）。"""
    def is_valid(base):
        b = Path(base)
        return ((b / "fortunelib" / "__init__.py").is_file()
                and (b / "data").is_dir())

    w = Path(working_root)
    if w.exists() and is_valid(w):
        return str(w)

    inp = Path(input_root)
    if not inp.exists():
        return None

    candidates = []
    stack = [(inp, 0)]
    while stack:
        cur, depth = stack.pop()
        if depth > max_depth:
            continue
        try:
            for child in sorted(cur.iterdir()):
                if not child.is_dir():
                    continue
                if is_valid(child):
                    candidates.append((depth + 1, str(child)))
                else:
                    stack.append((child, depth))
        except PermissionError:
            continue

    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0])
    return candidates[0][1]


BASE = _find_fortunelib()
if BASE is None:
    print("!! fortunelib が見つかりません。マウントされている入力を確認してください:")
    inp = Path("/kaggle/input")
    if inp.exists():
        for p in sorted(inp.iterdir()):
            print(f"   /kaggle/input/{p.name} （{'dir' if p.is_dir() else 'file'}）")
            if p.is_dir():
                for q in sorted(p.iterdir())[:10]:
                    print(f"      └ {q.name}")
    else:
        print("   /kaggle/input が存在しません（Add Input が行われていない可能性）")
    raise SystemExit("fortunelib が見つかりません（上記ツリーを確認してください）")

sys.path.insert(0, BASE)
os.environ.setdefault("FORTUNE_DATA_DIR", str(Path(BASE) / "data"))
print("✅ fortunelib を検出:", BASE)
print("   FORTUNE_DATA_DIR:", os.environ["FORTUNE_DATA_DIR"])

try:
    _version_path = Path(BASE) / "data" / "dataset_version.json"
    BUILD_ID = json.loads(_version_path.read_text(encoding="utf-8")).get(
        "build_id", "unknown")
except Exception:
    BUILD_ID = "unknown"

# ============================================================
# STEP 2.5: Notebook の版チェック（mts5.40）
# ============================================================
# ★mts5.35 の事故: cellN2_synthesis.py に動画カードを足したのに、手元の Kaggle
#   Notebook を取り込み直していなかったため、本番で**動画のコードが走らなかった**。
#   Dataset の New Version だけでは Notebook の中身は変わらない、という落とし穴。
#   ここで古さを検出する（見つからなければ黙っている・例外も出さない）。
try:
    from fortunelib.notebook_guard import check_notebook_freshness as _check_nb
    _check_nb(BASE)
except Exception as _e_nb:
    print(f"   （Notebook の版チェックはスキップ: {type(_e_nb).__name__}）")

RUN_ERRORS = []

# ============================================================
# STEP 3: Gemini API バックエンドの設定
# ============================================================
from fortunelib.backend import GeminiBackend, LazyBackend, _resolve_gemini_api_key  # noqa: E402

# ★ 2026-08-28: 408 Request Timeout は backend.py 本体でリトライ対象
#   （429/5xx と同じ指数バックオフ）。ノートブック側の monkeypatch は不要。

# ★ モデル（mts5.67・2026-10-14・ユーザー決定）: gemini-3.8-flash
#   Google が gemini-3.5-flash と gemini-3.7-flash を**非推奨**にしたため、既定を
#   gemini-3.8-flash へ変更。404・拒否のときの候補順は 3.8 → 3.6 → 3-flash-preview
#   （fortunelib/backend.py の gemini_model_candidates と model_plan.json が同じ順）。
#   非推奨の2モデルは一次・fallback・404候補のすべてから外してある（指定しないこと）。
#   thinking_level=minimal が 3.8 で拒否された場合は minimal → low → 除去 の順に
#   自動で緩めて再送する（backend.py）。
# ★ 本番の各段（Stage A〜D）のモデルは MODEL_NAME ではなく
#   fortunelib/data/model_plan.json（plan B5）で決まる（下の get_mts_backends()）。
#   MODEL_NAME は単体 GeminiBackend（セルフテスト・補助呼び出し）の既定。
# （履歴）2026-08-26: gemini-2.5-flash（2026-10-16 以降に廃止予定）→ gemini-3.5-flash。
#   2026-08-27: AI Studio 実測表＝3.5-flash は RPD 20/日、3.5-flash-lite は RPD 500/日。
#   ※ Pro 系は 2026-04 に無料枠から除外されたため指定しないこと。
#   ※ 404時は同一モデルファミリーの候補へ切替。実IDは Cell 0.1 の
#     疎通確認に表示される。Pro系・Gemini 2系・2.5系は一次モデルに指定しない。
MODEL_NAME = "gemini-3.8-flash"

# ★ 無料枠のレート制限は概ね 10〜15回/分。安全側の 10 に設定。
#   1回の完走は約67呼び出しなので、日次上限（250〜1,500回/日・モデルによる）
#   に対しても 1 日 1〜3 回は余裕で実行できる。
REQUESTS_PER_MINUTE = 10.0

# ============================================================
# GOOGLE_API_KEY の取得（3段階）
# ① 環境変数: Secrets を attach した編集モード実行で展開される
# ② kaggle_secrets: Save Version（Commit）実行でも動作する正攻法
# ③ None → ここで停止（キーが無いと何も動かないため、早期に止める）
# ============================================================
GOOGLE_API_KEY = _resolve_gemini_api_key()
if not GOOGLE_API_KEY:
    print("❌ GOOGLE_API_KEY が取得できません。以下の手順で設定してください:")
    print("   ① https://aistudio.google.com にアクセス →「Get API key」で無料発行")
    print("      （クレジットカード登録は不要）")
    print("   ② Kaggle 右側パネル → Add-ons → Secrets →「GOOGLE_API_KEY」の名前で登録")
    print("   ③ この Cell 0 を再実行")
    raise SystemExit("GOOGLE_API_KEY が未設定です（上記①〜③を実施してください）")
print("✅ GOOGLE_API_KEY を取得しました")


def load_backend():
    """MTS以前の単一モデル互換用（MTSでは get_mts_backends() を使用）。"""
    return GeminiBackend(model_name=MODEL_NAME, api_key=GOOGLE_API_KEY)


BACKEND = LazyBackend(load_backend)  # 後方互換用（MTS実行では get_mts_backends() を使用）

# ★ MTS B2: モデルごとに独立 backend。段階 A/B/C の実割り当ては
#   data/model_plan.json。既定 MODEL_NAME はセルフテストの疎通確認等にも使う。
MTS_BACKENDS = None


def get_mts_backends():
    """モデル計画の全モデル分 GeminiBackend を1つずつ作る（スレッド専有）。"""
    global MTS_BACKENDS
    if MTS_BACKENDS is None:
        from fortunelib.mts import load_model_plan, make_gemini_backend
        plan = load_model_plan()
        MTS_BACKENDS = {
            m: make_gemini_backend(m, api_key=GOOGLE_API_KEY, verbose=True)
            for m in plan["models"]
        }
        print("   MTS backends:", ", ".join(MTS_BACKENDS))
    return MTS_BACKENDS


print("✅ ブートストラップ完了")
print("   メインモデル: Gemini API /", MODEL_NAME)
print("   MTS: ON（model_plan.json B2割当: Stage A/B/C マルチモデル並列）")
print("   ※ アクセラレータは GPU 不要（None/CPU のままでOK）")
print("   ※ 1回の完走でおよそ67回の API 呼び出しを消費します")



## ▶ セルフテスト（CI相当・推奨）

Add Input した Dataset に対してユニットテストを実行します（GPU不要・約1分＋Gemini API 疎通1呼び出し）。
**ここが失敗したら以降のセル（約20〜30分・約67回のAPI呼び出し）は流さないでください。**
Dataset のアップロード漏れ・バージョン取り違え・依存パッケージの導入失敗・APIキー不備を検出します。

In [ ]:
# ============================================================
# Cell 0.1: セルフテスト（CI相当）
# ------------------------------------------------------------
# Add Input した Dataset に対してユニットテストを実行する。
# GPU は不要（約1分＋API疎通1回）。Cell 0（ブートストラップ）の後に実行すること。
#
# 目的:
#   1. Dataset のアップロード漏れ・バージョン取り違えを即座に検出する
#   2. 依存パッケージの導入失敗を検出する
#      （import に失敗するとテストが「0.2秒で大量エラー」になる既知の罠）
#   3. Gemini API への疎通（APIキー・モデル名・レート制限）を確認する
#
# ここで失敗したら、以降のセル（本番実行・約20〜30分）は流さないこと。
# ============================================================
import os
import sys
import time
import unittest
from pathlib import Path

# --- ROOT の解決 --------------------------------------------
# ★BUG-20: ノートブックのセルには __file__ が存在しないため、
#   以前の実装は NameError で即死していた。
#   Cell 0 が確定させた情報（sys.path / FORTUNE_DATA_DIR）を第一に使い、
#   無ければ /kaggle/input を再帰探索する（Dataset の階層は環境で変わる）。
def _is_valid(b):
    b = Path(b)
    return (b / "fortunelib" / "__init__.py").is_file() and (b / "data").is_dir()


def _resolve_root():
    # 1) Cell 0 が sys.path に入れた場所
    for p in sys.path:
        if p and _is_valid(p):
            return Path(p)
    # 2) Cell 0 が設定した FORTUNE_DATA_DIR の親
    dd = os.environ.get("FORTUNE_DATA_DIR")
    if dd and _is_valid(Path(dd).parent):
        return Path(dd).parent
    # 3) 定番の場所
    for p in (Path("/kaggle/working"), Path.cwd(), *Path.cwd().parents):
        if _is_valid(p):
            return Path(p)
    # 4) /kaggle/input を再帰探索（深さ優先・浅いものを採用）
    inp = Path("/kaggle/input")
    if inp.exists():
        found, stack = [], [(inp, 0)]
        while stack:
            cur, d = stack.pop()
            if d > 6:
                continue
            try:
                for c in sorted(cur.iterdir()):
                    if not c.is_dir():
                        continue
                    if _is_valid(c):
                        found.append((d + 1, c))
                    else:
                        stack.append((c, d + 1))
            except PermissionError:
                continue
        if found:
            found.sort(key=lambda x: x[0])
            return found[0][1]
    return None


def _selftest_fail(message, details=""):
    """セルフテストを停止する（以降のセルを流さないため SystemExit）。"""
    if details:
        print(details)
    raise SystemExit(1)


ROOT = _resolve_root()
if ROOT is None:
    print("❌ fortunelib が見つかりません。先に Cell 0 を実行し、")
    print("   Add Input で Dataset を追加しているか確認してください。")
    _selftest_fail("fortunelib が見つかりません。Dataset と Add Input を確認してください。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("FORTUNE_DATA_DIR", str(ROOT / "data"))
print(f"ROOT: {ROOT}")

# --- テストディレクトリの確認 --------------------------------
TESTS = ROOT / "tests"
if not TESTS.is_dir():
    print(f"⚠ tests/ が Dataset に含まれていません（{TESTS}）")
    print("  build_dataset.py に tests を含めて再アップロードすると")
    print("  このセルで回帰検査ができます。以降のセルは実行可能です。")
    raise SystemExit(0)

# --- 1. 依存パッケージの健全性 -------------------------------
# ★ここが肝。import に失敗したままテストを走らせると、
#   「0.2秒で全部エラー」という偽の失敗になり原因を見失う。
REQUIRED = ["sxtwl", "swisseph", "kinqimen", "lunar_python", "ephem",
            "google.genai"]
missing = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except ImportError:
        missing.append(mod)
if missing:
    print(f"❌ 依存パッケージが未導入: {missing}")
    print("   → Cell 0（ブートストラップ）を先に実行してください。")
    print('   pip install sxtwl pyswisseph iztro-py divicast lunar_python "ephem==4.2.1"')
    print("   pip install --no-deps kinqimen")
    print("   pip install google-genai   ← Gemini API 用")
    _selftest_fail(f"依存パッケージが未導入: {missing}")
print(f"✅ 依存パッケージ {len(REQUIRED)} 件 OK")

# --- 1b. Gemini API 疎通（軽量モデルで 1〜3 リクエスト）--------------
# ★ 本番（約67呼び出し）の前に、キー・モデル名・レート制限を少ない呼び出しで
#   検証しておく。上位 flash（gemini-3.8-flash・日次20回枠を仮定）を疎通で消費
#   しないよう、枠に余裕のある軽量モデルを使う。
# ★ v1.0.0（2026-10-15 実機）: 疎通専用だった gemini-3.1-flash-lite が Gemini 側の
#   混雑（503 "high demand"）を 3 回連続で返し、ここで止まって本番に進めなかった。
#   503/408 はキー・Dataset・Notebook の異常ではない（キー不正は 400/403、モデル名
#   違いは 404、日次枠切れは 429）。本番の MTS が混雑モデルを別モデルへ移譲するのと
#   同じ思想で、疎通も軽量モデル 3 本（backend.PING_MODELS）を順に試す。
#   判定ロジックは fortunelib.backend.ping_gemini（単体テストあり）に置く。
from fortunelib.backend import PING_MODELS, ping_gemini  # noqa: E402
_ping = ping_gemini()
if _ping.status == "no_key":
    print("⚠ GOOGLE_API_KEY が未設定のため API 疎通検査をスキップします")
    print("   （本番には必須です。Cell 0 の表示に従って Secrets を登録してください）")
elif _ping.ok:
    print(f"✅ Gemini API 疎通 OK（model={_ping.model}・"
          f"{_ping.seconds:.1f}秒・応答: {_ping.reply[:20]}）")
    if len(_ping.tried) > 1:
        _busy = "・".join(m for m, _r in _ping.tried[:-1])
        print(f"   ※ {_busy} は混雑中（503/408）でした。本番では混雑したモデルを"
              "自動で別モデルへ移譲しますが、混雑が続くと所要時間が延びます")
    print("   ※ MTS実行では各段階の担当モデル（3.1-lite/3.5-lite/3.8-flash/3.6-flash/3-flash-preview）が並列稼働します")
elif _ping.status == "empty":
    print(f"❌ Gemini API が空の応答を返しました（model={_ping.model}）")
    print("   考えられる原因:")
    print("   ・セーフティフィルタ（プロンプトを見直す）")
    print("   ・thinkingトークンが出力枠を使い切った")
    print("      可能性 → このログを開発側に報告してください")
elif _ping.status == "congested":
    print("❌ Gemini API が混雑しています（503/408）。本番セルは実行しないでください:")
    print(f"   軽量モデル {len(PING_MODELS)} 本（{'・'.join(PING_MODELS)}）がすべて混雑でした。")
    print("   これは Gemini 側の一時的な混雑で、API キー・Dataset・Notebook は正常です")
    print("   （キー不正なら 400/403、モデル名違いなら 404、日次枠切れなら 429 になります）。")
    print("   対処: 30〜60 分ほど待ってから、Cell 0 からやり直してください。")
    for _m, _r in _ping.tried:
        print(f"     - {_m}: {_r}")
else:
    print("❌ Gemini API 疎通に失敗しました。本番セルは実行しないでください:")
    print("  ", _ping.error)
# SystemExit は except ブロックの外で上げる（例外の連鎖で IPython の表示が崩れないように）
if _ping.status == "empty":
    _selftest_fail("Gemini API が空の応答を返しました")
elif _ping.status == "congested":
    _selftest_fail("Gemini API が混雑中（503/408）のため中断しました。時間をおいて再実行してください")
elif _ping.status == "fatal":
    _selftest_fail("Gemini API 疎通に失敗しました")

# --- 2. Dataset の中身が期待どおりか --------------------------
from fortunelib.registry import all_slugs                      # noqa: E402
from fortunelib.calc.deities import all_deities                # noqa: E402
from fortunelib.tone import TONE_DEFS                          # noqa: E402

n_slugs, n_deities = len(all_slugs()), len(all_deities())
n_tones = len(TONE_DEFS)
print(f"   占術数: {n_slugs} / 神格: {n_deities}柱 / トーン: {n_tones}種")
problems = []
if n_slugs != 53:
    problems.append(f"占術数が {n_slugs}（期待 53）")
if n_deities != 138:
    problems.append(f"神格数が {n_deities}（期待 138）")
# mts5.65: トーンの数は**版の印（dataset_version.json の expected.tones）を正本**にする
# （obaa の削除などでトーンを増減しても、Notebook 側の書き換えが要らないように）。
# 印が無い古い Dataset では、この版の実数（3＝standard/easy/spiritual）を期待値にする。
_TONES_DEFAULT = 3
# --- 2b. バージョン印との照合（古い Dataset の検出）-------------
# ★BUG-22: Dataset を更新せずに実行すると、古いコードで本番が走ってしまう。
#   テストの合否より前に、Dataset の鮮度そのものを検査する。
import json as _json                                            # noqa: E402
_expected_tests = None          # mts5.52: expected.tests を「実行件数」と突合する
vf = ROOT / "data" / "dataset_version.json"
if not vf.is_file():
    problems.append("dataset_version.json が無い（Dataset が古い可能性）")
else:
    v = _json.loads(vf.read_text(encoding="utf-8"))
    print(f"   build_id: {v.get('build_id', '?')}")
    exp = v.get("expected", {})
    if exp.get("slugs") != n_slugs:
        problems.append(f"占術数が印と不一致（印{exp.get('slugs')}／実{n_slugs}）")
    if exp.get("deities") != n_deities:
        problems.append(
            f"神格数が印と不一致（印{exp.get('deities')}／実{n_deities}）")
    _exp_tones = int(exp.get("tones", _TONES_DEFAULT))
    if _exp_tones != n_tones:
        problems.append(f"トーン数が印と不一致（印{_exp_tones}／実{n_tones}）")
    # mts5.52（F-5）: 期待テスト数（dataset_version.json の expected.tests）を控えておき、
    # 実行後の件数と突合する。従来は**どのコードもこの値を読んでおらず**、
    # テストが減っていても「実行できた分が全部成功」なら合格していた。
    _expected_tests = exp.get("tests")
    if isinstance(_expected_tests, int):
        print(f"   期待テスト数: {_expected_tests}（実行後に件数を照合します）")
    # 入力は people_sample.csv のみ（cell1_input のハードコードは廃止）。
    # CSV が同梱されていることを確認する。
    csv_path = ROOT / "kaggle" / "people_sample.csv"
    if not csv_path.is_file():
        problems.append("people_sample.csv が Dataset に含まれていません")

if problems:
    print("❌ Dataset が最新版ではない可能性があります:")
    for x in problems:
        print(f"   - {x}")
    print("   → build_dataset.py --zip で作り直し、Dataset を更新（New Version）し、")
    print("     ノートブックの Add Input を最新バージョンに切り替えてください。")
    _selftest_fail("Dataset のバージョン／内容の照合に失敗しました", "\\n".join(problems))

# --- 3. ユニットテスト --------------------------------------
t0 = time.time()
suite = unittest.TestLoader().discover(str(TESTS), top_level_dir=str(ROOT))
result = unittest.TextTestRunner(verbosity=1).run(suite)
elapsed = time.time() - t0

n = result.testsRun
print("\n" + "=" * 56)
print(f"テスト {n} 件 / {elapsed:.1f}秒 "
      f"（失敗 {len(result.failures)} / エラー {len(result.errors)}）")

# mts5.54: テストが積んだ警告を**ここで消す**。
# 実機（mts5.53）では、テストが製品コードを呼ぶため「⚠ 画像を埋め込めなかった占術: …」などの
# 警告が台帳（fortunelib.runcheck）に残り、**本番セル（Cell 1・N2）の警告に混ざる**状態でした
# （`RUN_STRICT=1` なら本番セルがテスト由来の警告で不合格になってしまう）。セルごとに独立させます。
try:
    from fortunelib.runcheck import clear as _warn_clear
    _warn_clear()
    print("※ 警告台帳をリセットしました（テスト由来の警告を本番セルへ持ち越しません）")
except Exception:
    pass

# 0.2秒台で終わるのは import 崩壊のサイン（実行時間そのものを検査する）
if n > 100 and elapsed < 5:
    print("❌ 実行が速すぎます。import 崩壊の可能性が高いです")
    _selftest_fail("セルフテストの実行が速すぎます（import 崩壊の可能性）")

# mts5.52（F-5）: 実行件数が期待値に足りなければ不合格（テストが黙って減る事故の防止）
if isinstance(_expected_tests, int) and n < _expected_tests:
    print(f"❌ テスト実行数が期待より少ない: 実行 {n} < 期待 {_expected_tests}")
    print("   → Dataset の tests/ が古い（テストが減った）可能性があります。")
    print("      build_dataset_zip.py で作り直し、Dataset を New Version に更新してください。")
    _selftest_fail(
        f"セルフテストの実行件数が期待値に足りません（実行 {n} / 期待 {_expected_tests}）",
        "tests/ の不足・収集エラーの可能性。Dataset を作り直してください。")

if not result.wasSuccessful():
    print("❌ テスト失敗。本番セルは実行しないでください")
    _selftest_fail(
        f"セルフテスト失敗: failures={len(result.failures)}, errors={len(result.errors)}",
        "\\n".join(str(x[0]) for x in result.failures + result.errors)[:4000],
    )

print("✅ セルフテスト合格。以降のセルを実行して問題ありません")
print("=" * 56)


## ▶ 入力（CSV）＋ MTS Stage A

**入力は people_sample.csv だけです。** このセルで CSV の全員分の Stage A（個別占術52種）を生成します。Stage B/C は Cell N2 で実行します。

列: `person_id, birth_date, birth_time, birth_place, gender, blood_type, name_kanji, name_kana, partner_date, partner_time, partner_place, consultation, target`

- `consultation` を空欄にすると、その人は「今日の運勢モード」で実行されます。
- `partner_*` を埋めると相性系2種（シナストリー・パリミチャム）が実行されます。
- 人物を追加・差し替えるときは CSV を書き換えるだけです。

In [ ]:
# ============================================================
# Cell 1: 入力（people_sample.csv を読んで MTS Stage A を実行）
# ------------------------------------------------------------
# 入力は people_sample.csv だけです。このセルを実行すると、CSV の
# 全員分の Stage A（個別占術 52 種）を生成します。
# Stage B/C（統合レポート・御縁のある神さま）は Cell N2 で実行します。
# ============================================================
from pathlib import Path  # noqa: E402

from fortunelib.batch_runner import load_csv  # noqa: E402

# ---- 設定 ---------------------------------------------------
# CSV の置き場所（Dataset に同梱 / /kaggle/working に自分でアップロード のいずれか）
CSV_CANDIDATES = [
    "/kaggle/working/people.csv",                              # 自分でアップロードした場合
    "/kaggle/input/fortune_dataset/kaggle/people_sample.csv",  # 最新 Dataset 配布名
    "/kaggle/input/fortune_dataset/people_sample.csv",          # 旧パス互換
    "/kaggle/input/fortune-dataset/kaggle/people_sample.csv",  # ハイフン名
    "/kaggle/input/fortune-dataset/people_sample.csv",          # ハイフン名互換
    "kaggle/people_sample.csv",                                # ローカル実行
    "people_sample.csv",                                       # ローカル互換
]
OUT_ROOT = "/kaggle/working/output"   # 出力先（person_id ごとにサブフォルダ）
LIMIT = None                          # 先頭N人だけ試す場合は数値（例: 2）
#   ※ 統合レポート＋神々＋HTMLまで含めると1人あたり約67回のAPI呼び出し
#      （所要20〜30分・無料枠の日次上限に収まるのは1〜3人程度）。

# ---- CSV を探す ---------------------------------------------
CSV_PATH = None
for _c in CSV_CANDIDATES:
    if Path(_c).exists():
        CSV_PATH = _c
        break
if CSV_PATH is None:
    # Dataset 配下を再帰探索（マウントパスの違いを吸収）
    for _p in Path("/kaggle/input").rglob("*.csv"):
        if "people" in _p.name:
            CSV_PATH = str(_p)
            break

if CSV_PATH is None:
    _csv_error = (
        "CSVが見つかりません。/kaggle/working/people.csv にアップロードするか、\n"
        "Dataset に people_sample.csv を含めてください。\n"
        "列: person_id, birth_date, birth_time, birth_place, gender, blood_type,\n"
        "    name_kanji, name_kana, partner_date, partner_time, partner_place,\n"
        "    consultation, target"
    )
    raise FileNotFoundError(_csv_error)

print("CSV:", CSV_PATH)
_rows = load_csv(CSV_PATH)
print(f"対象 {len(_rows)}名")
for _r in _rows:
    _mode = "今日の運勢モード" if not (_r.get("consultation") or "").strip() else "相談あり"
    _pt = "相手あり" if (_r.get("partner_date") or "").strip() else "相手なし"
    print(f"  - {_r.get('person_id') or '(自動採番)':16s} {_r['birth_date']}  {_mode} / {_pt}")

# ---- 実行 ---------------------------------------------------
# mts5.54: 前のセル（セルフテスト）の警告を持ち越さない。このセルの警告だけを数える。
# mts5.55: 再利用の内訳（SKIP_STATS）も同じくプロセス共通なので、ここで0に戻す
#          （セルフテストで起きた再利用が「本番で再利用した」ように見える誤表示を防ぐ）。
try:
    from fortunelib.runcheck import clear as _warn_clear
    from fortunelib.pipeline import reset_skip_stats as _skip_reset
    _warn_clear()
    _skip_reset()
except Exception:
    pass
# MTS Stage A のみをここで実行し、Stage B/C は Cell N2 が担当する。
# 新しいCSV実行として、前回の通知エラーを持ち越さない。
RUN_ERRORS = []
from fortunelib.mts import run_person_mts  # noqa: E402
_rows2 = _rows[:LIMIT] if LIMIT else _rows
RESULTS = []
_be = get_mts_backends()
for _row in _rows2:
    RESULTS.append(run_person_mts(
        _row, _be, Path(OUT_ROOT), stages="A", verbose=True))

# Stage A の未解決エラーは、Cell N2 の最終通知にも引き継ぐ。
_run_errors = globals().setdefault("RUN_ERRORS", [])
for _r in RESULTS:
    if _r.error:
        _run_errors.append({
            "stage": "A", "target": _r.person_id,
            "message": f"個別占術の失敗 {_r.error} 件: {_r.failed[:12]}",
        })

# ★統合レポート（Cell N2）が、このCSVバッチの結果を対象に選べるよう
#   フラグを立てる（Cell N2 の出力先判定に使用）。
CSV_BATCH_DONE = True

# mts5.53（P-10）: 段階Aの警告（再利用・照合の欠落・フォントなど）を1か所にまとめる。
# 既定は警告のみで実行は続く。RUN_STRICT=1 のときだけ不合格（SystemExit(1)）にする。
try:
    from fortunelib.runcheck import WARNINGS as _A_WARNINGS, enforce as _a_enforce, report as _a_report
    if _A_WARNINGS:
        print("\n■ 警告（既定では合否に影響しません）")
    _a_report(prefix="  ⚠")
    _a_enforce()
except SystemExit:
    raise
except Exception as _a_e:
    print(f"  ⚠ 警告の集約に失敗（{type(_a_e).__name__}: {_a_e}）")

print("\n" + "=" * 56)
print("■ バッチ完了")
for _r in RESULTS:
    print(f"  {_r.person_id:16s} 生成{_r.ok:3d} / 対象外{_r.unavailable} / 失敗{_r.error}")
print(f"\n→ 出力: {OUT_ROOT}/  （人ごとのフォルダに結果一式。一覧の index.md はありません）")

print("\n※ 統合レポート（Cell N2）は各 person フォルダに対して生成されます")
print("※ 中断・再開する場合は、Cell N2 完了時に保存される Dataset チェックポイントを")
print("   次回 Add Input に追加してください（結果はスキップガードで再利用されます）")


## ▶ 統合鑑定レポート（3段階）

Stage Aの全占術結果を **クラスタ単位で中間レポート（950〜2,100字×11本）** にまとめ、
**軸サマリー2本** を経て **最終レポート（3,400〜4,800字）** を作成します。

- `synthesis_report.md` … 読み物としての統合レポート
- `cluster_reports/` … 分野別の中間レポート
- `final_report.md` … 従来の索引（実行ログ的な一覧・そのまま残ります）

冒頭の固定動画（Dataset 同梱 `assets/video/intro_video.json`）は Cell N2 の `EMBED_INTRO_VIDEO` でON/OFFできます（既定 `True`）。ON なら HTML の目次の直前に動画カード（Base64 埋め込み・自動再生なし・オンライン版 YouTube への案内つき）が入り、HTML は約 +10 MB になります。

MTS B5の固定割当で処理し、Stage Cは `gemini-3.8-flash`（軸2本＋最終1本・予備 3.6-flash → 3-flash-preview）および `gemini-3-flash-preview`（御縁のある神さま Top10）で実行します（非推奨の 3.5-flash／3.7-flash は不使用）。
所要時間の目安: Gemini Flash で約4〜7分（中間11本＋軸2本＋最終1本＋神格選定）。

In [ ]:
# ============================================================
# Cell N2: 統合鑑定レポート → 御縁のある神さま → HTML一括出力
#
#   ① クラスタ中間レポート（1000〜2000字 × 5本＋6本）
#   ② 軸サマリー（method / culture の各総括）
#   ③ 最終統合レポート（両軸を突き合わせ・約4000字）
#   ④ 御縁のある神さま Top10（③の内容を踏まえて選定 → ③に追記）
#   ⑤ 全成果物を1枚のHTMLに束ねる（スマホ保存・閲覧用）
# ============================================================
import json  # noqa: E402
from pathlib import Path  # noqa: E402

from fortunelib.batch import build_profile_facts, build_calc_results  # noqa: E402
from fortunelib.html_report import build_html  # noqa: E402
from fortunelib.pipeline import PipelineContext  # noqa: E402
from fortunelib.profile import normalize_input  # noqa: E402
from fortunelib.mts import run_stage_b_mts, run_stage_c_mts  # noqa: E402

# ---- 設定 ---------------------------------------------------
# mts5.54: 前のセル（セルフテスト・段階A）の警告を持ち越さない。このセルの警告だけを数える。
# mts5.55: 再利用の内訳（SKIP_STATS）も同じくプロセス共通なので、ここで0に戻す。
try:
    from fortunelib.runcheck import clear as _warn_clear
    from fortunelib.pipeline import reset_skip_stats as _skip_reset
    _warn_clear()
    _skip_reset()
except Exception:
    pass
MAKE_HTML = True                       # 1枚HTMLを生成する
EMBED_INTRO_VIDEO = True               # ON/OFF: True なら同梱の冒頭動画（assets/video/intro_video.json）を目次の前に埋め込む（既定ON）

# 出力先の決定（CSVバッチ後か、1人モードかを自動判定）
_OUT_ROOT = Path("/kaggle/working/output")
if globals().get("CSV_BATCH_DONE") and _OUT_ROOT.is_dir():
    TARGETS = [p for p in sorted(_OUT_ROOT.iterdir()) if (p / "results").is_dir()]
else:
    TARGETS = [Path("/kaggle/working")]

print(f"■ MTS 統合レポート  対象 {len(TARGETS)}件 / 軸: method + culture")

_RUN_ERRORS = globals().setdefault("RUN_ERRORS", [])
_PROCESSED_TARGETS = 0
_SKIPPED_TARGETS = []

for _dir in TARGETS:
    _rd = _dir / "results"
    if not _rd.is_dir() or not list(_rd.glob("*_raw.json")):
        print(f"− {_dir.name}: 結果がないためスキップ")
        _SKIPPED_TARGETS.append(str(_dir))
        continue

    try:
        # プロファイル・相談内容を復元
        _sp = _dir / "profile_summary.json"
        if _sp.exists():
            _s = json.loads(_sp.read_text(encoding="utf-8"))
            _facts, _consult = _s.get("facts", {}), _s.get("consultation", "")
        else:
            _s = {}
            _facts = globals().get("FACTS") or {}
            _consult = globals().get("NI").consultation if globals().get("NI") else ""

        # 神々の生成に使う PipelineContext を用意
        #   CSVバッチ後やカーネル再起動後でも、profile_summary.json から確実に復元する
        _ctx = None
        _ni_for_target = None
        _calc_for_target = globals().get("CALC") or {}
        try:
            if not globals().get("CSV_BATCH_DONE") and globals().get("CTX") is not None:
                _ctx = CTX
            if _ctx is None and _sp.exists():
                _inp = _s.get("input_raw")
                _ni2 = normalize_input(_inp) if _inp else None
                _ni_for_target = _ni2
                if _ni2 is not None:
                    _f2 = build_profile_facts(_ni2)
                    _c2 = build_calc_results(_ni2, consultation=_ni2.consultation)
                    _calc_for_target = _c2
                    _ctx = PipelineContext(
                        inputs=_ni2, profile_facts=_f2, calc_results=_c2,
                        consultation=_ni2.consultation, results_dir=_rd,
                        summary_path=_sp, backend=BACKEND)
        except Exception as _e:
            raise RuntimeError(f"神々の実行コンテキストを用意できませんでした: {_e}") from _e

        # ★ F1ガード: _ctx が用意できない場合は絶対に沈黙スキップせず例外で落とす
        if _ctx is None:
            raise RuntimeError(
                f"{_dir.name}: 神々の実行コンテキスト (_ctx) を復元できませんでした。"
                f"profile_summary.json または Cell 1 の実行状態を確認してください。")

        _tgt = ""
        _cd = (_calc_for_target or {}).get("target_date") or {}
        # mts4.1表示層ゲート: 相談あり＋対象日空欄（実行日フォールバック）なら
        # Stage B/C へ対象日を渡さない。今日の運勢モードか明示指定なら渡す。
        # 明示指定は inputs（または input_raw）から導出し、calc スキーマは不変。
        if _ni_for_target is not None:
            _explicit_t = _ni_for_target.target_date is not None
        else:
            _explicit_t = bool(((_s.get("input_raw") or {}).get("target") or "").strip())
        if isinstance(_cd, dict) and (not _consult.strip() or _explicit_t):
            _tgt = _cd.get("date", "")

        print(f"\n▶ {_dir.name}")
        print("  ▶ MTS 段階 B（クラスタ lite 並列）")
        _b = run_stage_b_mts(
            _rd, _dir, get_mts_backends(), _facts,
            consultation=_consult, target_date=_tgt, verbose=True)
        _bcheck = _b.get("verification") or {}
        print(f"  ✅ MTS 段階 B 検証: {_bcheck.get('found', 0)}/"
              f"{_bcheck.get('expected', 0)}本・fallback "
              f"{len(_b.get('fallbacks') or [])}件")
        if not _b.get("complete") or _bcheck.get("found") != _bcheck.get("expected"):
            raise RuntimeError(
                f"Stage B が {_bcheck.get('found', 0)}/{_bcheck.get('expected', 0)}本です")
        print("  ▶ MTS 段階 C（最終＋神々・最上位 flash）")
        _res = run_stage_c_mts(
            _rd, _dir, get_mts_backends(), _facts,
            consultation=_consult, target_date=_tgt, verbose=True,
            ctx=_ctx, per_axis=_b.get("per_axis"))

        # ★ F1ガード: 神々の節が生成され、かつ正常であることを厳格に検証
        _deities_res = _res.get("deities")
        if not _deities_res or _deities_res.get("status") != "ok":
            _dstatus = (_deities_res or {}).get("status") or "未実行"
            _dreason = (_deities_res or {}).get("reason") or ""
            raise RuntimeError(
                f"神々の生成に失敗しました（status={_dstatus}, reason={_dreason}）。"
                f"最終レポートに御縁のある神さまが追記されていません。")

        print(f"  ✅ 最終 {len(_res['final_report'])}字 → {_res['file']}")
        from fortunelib.mts import deities_model
        print(f"  ✅ 御縁のある神さま: {deities_model()} で追記済み")

        # mts5.22: Stage D トーン化（かんたん／スピリチュアル風・mts5.65 で obaa を削除）
        # 定型本文を凍結した文体リライト。失敗しても「標準」のみのHTMLを生成して継続する
        # （UX機能の本番鑑定への影響は許さない＝失敗は警告のみ）。
        try:
            from fortunelib.tone import run_stage_d_tones
            _tone_man = run_stage_d_tones(
                _rd, _dir, get_mts_backends(), _facts, verbose=True)
            _tsum = ", ".join(
                f"{_tid}: {_tm.get('n_ok')}/{_tm.get('n_total')}({_tm.get('status')})"
                for _tid, _tm in (_tone_man.get("tones") or {}).items())
            print(f"  ✅ Stage D トーン化: {_tsum or '未生成'}")
        except Exception as _te:
            print(f"  ⚠ Stage D（トーン化）に失敗 → 標準のみのHTMLで継続: "
                  f"{type(_te).__name__}: {_te}")

        if MAKE_HTML:
            # mts5.53（P-4）: 型Cの画像は同梱フォントで描く。フォントが欠けると
            # 例外も警告も出ず画像だけが変わる（実証済み）ため、ここで必ず確認する。
            try:
                from fortunelib.chartdraw import check_bundled_fonts as _check_fonts
                _check_fonts(verbose=True)
            except Exception as _e_font:
                print(f"  ⚠ フォントの確認に失敗（{type(_e_font).__name__}: {_e_font}）")
            # 本番埋め込み: assets/images の各デッキから、結果に対応する1枚ずつを選抜して Base64 埋め込み
            # 画像ディレクトリの探索は fortunelib.html_report.discover_assets_root に一本化
            # （順番: Dataset ルート（BASE）→ 環境変数 FORTUNE_ASSETS_DIR / FORTUNE_DATA_DIR →
            #   /kaggle/input/**/assets/images の glob → /kaggle/input 直下の列挙。個人のパスは置かない）
            from fortunelib.html_report import discover_assets_root as _discover_assets_root
            _images_dir = _discover_assets_root(globals().get("BASE"))
            # mts5.53（P-8）: 監査の指摘＝画像ディレクトリの探索に失敗しても
            # **無言で**画像なしの HTML が出来ていた（どの候補を見たかも残らない）。
            # ここで必ず結果を出し、失敗は警告として残す。
            if _images_dir is not None:
                print(f"  [画像] 素材ディレクトリ: {_images_dir}")
            else:
                _msg_img = ("画像素材ディレクトリ（assets/images）が見つかりませんでした。"
                            "画像なしで HTML を作ります（Add Input の Dataset に "
                            "assets/images が含まれているか確認してください）")
                print(f"  ⚠ {_msg_img}")
                try:
                    from fortunelib.runcheck import add_warning as _add_warning
                    _add_warning(_msg_img)
                except Exception:
                    pass

            # 同梱の固定動画（冒頭カード・目次の直前・全員共通）。
            # OFF なら読み込みも埋め込みもしない。無ければ警告を残して動画なしで続ける。
            _intro_html, _intro_info = "", {"reason": "disabled"}
            if EMBED_INTRO_VIDEO:
                try:
                    from fortunelib.intro_video import load_intro_video_card as _load_iv
                    _intro_html, _intro_info = _load_iv(_images_dir or globals().get("BASE"))
                except Exception as _e_iv:
                    _intro_info = {"reason": "error", "error": f"{type(_e_iv).__name__}: {_e_iv}"}
                if _intro_html:
                    print(f"  ✅ 冒頭動画カード: {_intro_info.get('file')}"
                          f"（{_intro_info.get('size_mb', 0):.2f}MB・約{_intro_info.get('seconds', 0):.0f}秒・"
                          f"SHA一致 {'OK' if _intro_info.get('sha_match') else '未確認'}・"
                          f"オンライン案内 {'有' if _intro_info.get('online_url') else '無'}）")
                else:
                    _msg_iv = ("冒頭動画カードを埋め込めませんでした"
                               f"（{_intro_info.get('reason')}: "
                               f"{_intro_info.get('error') or _intro_info.get('file') or _intro_info.get('searched') or ''}）"
                               "。動画なしで HTML を作ります（Dataset の assets/video に intro_video.json と MP4 があるか確認）")
                    print(f"  ⚠ {_msg_iv}")
                    try:
                        from fortunelib.runcheck import add_warning as _add_warning_iv
                        _add_warning_iv(_msg_iv)
                    except Exception:
                        pass
            else:
                print("  ⏭ 冒頭動画カードはOFF（EMBED_INTRO_VIDEO=False）のためスキップ（HTML は通常どおり作成）")

            _html = build_html(_dir, person_id=_dir.name, images_dir=_images_dir,
                               intro_video_html=_intro_html)
            if _images_dir:
                # 表示: 実際に埋め込まれた枚数と、画像を持つデッキ数を数える
                # （固定のツール一覧だと陳腐化するため、生成済みHTMLと画像ディレクトリから集計する）
                _n_img, _n_deck = 0, 0
                try:
                    # mts5.69: 冒頭動画カードのサムネイルは数えない（占術画像ではない）
                    _n_img = (_html.read_text(encoding="utf-8").count("data:image")
                              - _intro_html.count("data:image"))
                    _root = _images_dir if _images_dir.name == "images" else _images_dir.parent
                    for _d in sorted(_root.iterdir()):
                        if _d.is_dir() and any(_f.suffix.lower() in (".webp", ".jpg", ".jpeg", ".png")
                                               for _f in _d.iterdir()):
                            _n_deck += 1
                except Exception:
                    pass
                # 簡易表示
                # mts5.55: ここで数えているのは**素材ディレクトリの数**（結果の対応先ではない）。
                # build_html 側の「対象 51デッキ（結果の対応先）」と混同しないよう書き分ける。
                print(f"  ✅ HTML: {_html}（{_html.stat().st_size/1024:.0f} KB）"
                      f"［画像埋め込み: {_images_dir}（{_n_img}枚 / 素材 {_n_deck}デッキ＝画像ディレクトリ）］")
            else:
                print(f"  ✅ HTML: {_html}（{_html.stat().st_size/1024:.0f} KB）")
        _PROCESSED_TARGETS += 1
    except Exception as _e:
        _message = f"{type(_e).__name__}: {_e}"
        _RUN_ERRORS.append({"stage": "N2", "target": str(_dir),
                            "message": _message})
        print(f"  ❌ {_dir.name}: {_message}")
        continue

if not _PROCESSED_TARGETS and not _RUN_ERRORS:
    _RUN_ERRORS.append({"stage": "N2", "target": "all",
                        "message": "処理対象の結果ファイルがありません"})

# mts5.53（P-10）: ここまでの警告（フォント・画像・再利用など）を1か所にまとめて出す。
# 既定は**警告のみ**で実行は続く（従来の運用を壊さない）。厳格に止めたいときだけ
# RUN_STRICT=1 を設定する（Kaggle なら先頭のセルで %env RUN_STRICT=1）。
try:
    from fortunelib.runcheck import WARNINGS as _WARNINGS, enforce as _enforce, report as _report
    if _WARNINGS:
        print("\n■ 警告（既定では合否に影響しません）")
    _report(prefix="  ⚠")
    _enforce()
except SystemExit:
    raise
except Exception as _e_warn:
    print(f"  ⚠ 警告の集約に失敗（{type(_e_warn).__name__}: {_e_warn}）")

print("\n■ 完了")
print("  synthesis_report.md  … 統合レポート（神々の節を含む）")
print("  cluster_reports/     … 軸サマリー＋分野別の中間レポート")
print("  *_report.html        … 全内容を1枚にまとめたHTML（スマホ保存用）")

if _RUN_ERRORS:
    _detail = "\n".join(
        f"[{e.get('stage')}] {e.get('target')}: {e.get('message')}"
        for e in _RUN_ERRORS[-20:])
    raise RuntimeError("占術シミュレーターがエラーで終了しました。上のログを確認してください。\n"
                       + _detail)

print(f"✅ 全工程が正常終了しました（{_PROCESSED_TARGETS}件）")
